<a href="https://colab.research.google.com/github/basersalm24/cookbook/blob/main/quickstarts/Get_started_OpenAI_Compatibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2025 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Getting started with the Gemini API OpenAI compatibility

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_OpenAI_Compatibility.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This example illustrates how to interact with the [Gemini API](https://ai.google.dev/gemini-api/docs) using the [OpenAI Python library](https://github.com/openai/openai-python).

This notebook will walk you through:

* Perform basic text generation using Gemini models via the OpenAI library
* Experiment with multimodal interactions, sending images on your prompts
* Extract information from text using structured outputs (ie. specific fields or JSON output)
* Use Gemini API tools, like function calling
* Generate embeddings using Gemini API models

More details about this OpenAI compatibility on the [documentation](https://ai.google.dev/gemini-api/docs/openai).

## Setup

### Install the required modules

While running this notebook, you will need to install the following requirements:
- The [OpenAI python library](https://pypi.org/project/openai/)
- The pdf2image and pdfminer.six (and poppler-utils as its requirement) to manipulate PDF files

In [7]:
%pip install -U -q openai pillow pdf2image pdfminer.six
!apt -qq -y install poppler-utils # required by pdfminer

poppler-utils is already the newest version (22.02.0-2ubuntu0.10).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


## Get your Gemini API key

You will need your Gemini API key to perform the activities part of this notebook. You can generate a new one at the [Get API key](https://aistudio.google.com/app/apikey) AI Studio page.

In [8]:
from openai import OpenAI

try:
  # if you are running the notebook on Google Colab
  # and if you have saved your API key in the
  # Colab secrets
  from google.colab import userdata

  GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

except:
  # enter manually your API key here if you are not using Google Colab
  GOOGLE_API_KEY = "--enter-your-API-key-here--"

# OpenAI client
client = OpenAI(
    api_key=GOOGLE_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

## Define the Gemini model to be used

You can start by listing the available models using the OpenAI library.

In [9]:
models = client.models.list()
for model in models:
  if 'gemini-2' in model.id:
    print(model.id)

models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash
models/gemini-2.5-flash-lite-preview-06-17
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-2.0-flash-thinking-exp-01-21
models/gemini-2.0-flash-thinking-exp
models/gemini-2.0-flash-thinking-exp-1219
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image-preview
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-2.5

## Define the Gemini model to be used

In this example, you will use the `gemini-2.0-flash` model. For more details about the available models, check the [Gemini models](https://ai.google.dev/gemini-api/docs/models/gemini) page from the Gemini API documentation.

In [10]:
MODEL_ID = "gemini-2.5-flash" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash-lite-preview-09-2025", "gemini-2.5-flash", "gemini-2.5-flash-preview-09-2025", "gemini-2.5-pro"] {"allow-input":true, isTemplate: true}

## Initial interaction - generate text

For your first request, use the OpenAI SDK to perform text generation with a text prompt.

In [12]:
from IPython.display import Markdown

prompt = "What is generative AI?" # @param

response = client.chat.completions.create(
  model=MODEL_ID,
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {
      "role": "user",
      "content": prompt
    }
  ]
)

Markdown(response.choices[0].message.content)

**Generative AI** is a category of artificial intelligence models that are capable of **producing new, original content or data** rather than just classifying or analyzing existing data.

Think of it this way:

*   **Traditional AI** (or "discriminative AI") might tell you if an image is a cat or a dog, or predict tomorrow's weather. It discriminates between different options.
*   **Generative AI** doesn't just recognize a cat; it can *create* a new, unique image of a cat that has never existed before, based on what it has learned about "cat-ness."

**How it Works (Simplified):**

1.  **Learning from Data:** Generative AI models are trained on vast datasets of existing content (e.g., millions of images, billions of text documents, hours of audio).
2.  **Identifying Patterns:** During training, they learn the underlying patterns, structures, styles, and relationships within that data. They essentially learn the "grammar" or "rules" of the data. For text, it learns language structure; for images, it learns shapes, colors, textures, and object relationships.
3.  **Generating New Content:** Once trained, when given a prompt or input, the model uses its learned understanding to create new output that is similar in style and quality to its training data, but genuinely novel and not just a copy.

**Key Characteristics and Capabilities:**

*   **Novelty:** It creates content that is unique and hasn't existed before.
*   **Diversity:** It can generate a wide range of variations on a theme.
*   **Coherence:** The generated content often makes logical sense and is internally consistent.
*   **Contextual Understanding:** Many models can understand and respond to complex prompts, integrating various elements.
*   **Multimodality:** Modern generative AI can often work across different types of data (e.g., generating an image from text, or text from an image).

**Common Types of Content Generated:**

*   **Text:** Articles, summaries, emails, code, poetry, scripts, chatbots (e.g., ChatGPT, Bard).
*   **Images:** Artwork, photorealistic scenes, logos, product designs (e.g., DALL-E, Midjourney, Stable Diffusion).
*   **Audio:** Music, voice cloning, sound effects, synthetic speech.
*   **Video:** Deepfakes, animated characters, short clips (still an emerging but rapidly advancing area).
*   **Code:** Generating code snippets, functions, or even entire programs (e.g., GitHub Copilot).
*   **3D Models:** Creating 3D objects or environments from text descriptions.

**Underlying Technologies (Examples):**

While there are many architectures, some prominent ones include:

*   **Generative Adversarial Networks (GANs):** Involve two neural networks, a "generator" that creates content and a "discriminator" that tries to tell if it's real or fake, pushing each other to improve.
*   **Variational Autoencoders (VAEs):** Learn to encode data into a compressed representation and then decode it back into a new sample.
*   **Transformers:** Especially powerful for sequential data like text, these models are at the heart of large language models (LLMs) and are increasingly used for images and other modalities.
*   **Diffusion Models:** Currently very popular for image generation, these models learn to reverse a process of gradually adding noise to data, effectively "denoising" random inputs into coherent images.

**Why is it significant?**

Generative AI represents a major leap in AI capabilities, democratizing creativity, automating complex tasks, and opening up entirely new possibilities across various industries, from art and design to medicine and software development. It's rapidly changing how we interact with technology and create digital content.

### Generating code

You can work with the Gemini API to generate code for you.

In [13]:
prompt = """
    Write a C program that takes two IP addresses, representing the start and end of a range
    (e.g., 192.168.1.1 and 192.168.1.254), as input arguments. The program should convert this
    IP address range into the minimal set of CIDR notations that completely cover the given
    range. The output should be a comma-separated list of CIDR blocks.
"""

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

Markdown(response.choices[0].message.content)

The C program below takes two IP addresses as command-line arguments, representing a start and an end IP of a range. It then converts this range into the minimal set of CIDR notations that completely cover all IP addresses within the given range (inclusive). The output is a comma-separated list of these CIDR blocks.

### Algorithm Explanation

The core algorithm is a greedy approach that iteratively finds the largest possible CIDR block starting from the current IP address (`current_ip`) that:

1.  **Is aligned:** The `current_ip` must be a valid network address for the chosen prefix length (`p`). This means the last `(32 - p)` bits of `current_ip` must be zero.
2.  **Fits the range:** The entire CIDR block (from `current_ip` to `current_ip + block_size - 1`) must not exceed the `end_ip_num` of the input range.

The program proceeds as follows:

1.  **Input Parsing:** It takes two IP address strings (e.g., "192.168.1.1") from the command line.
2.  **IP Conversion:** It converts these IP strings into `uint32_t` (unsigned 32-bit integers) in host byte order using `inet_pton` and `ntohl`. This allows for easy arithmetic operations.
3.  **Range Validation:** It ensures that `start_ip_num` is less than or equal to `end_ip_num`, swapping them if necessary.
4.  **CIDR Generation Loop:**
    *   It initializes `current_ip` to `start_ip_num`.
    *   It enters a `while` loop that continues as long as `current_ip` is within or equal to `end_ip_num`.
    *   **Finding the best `prefix_len`:**
        *   Inside the loop, it iterates `p` (prefix length) from `32` down to `0`. This means it first tries to find `/32` blocks (single IP), then `/31` (2 IPs), `/30` (4 IPs), and so on, up to `/0` (all IPv4 addresses).
        *   For each `p`, it calculates the `block_size = 1ULL << (32 - p)`. `1ULL` is used to ensure `block_size` is an `unsigned long long` to correctly handle `2^32` for `/0`.
        *   It checks for alignment: `(current_ip & (block_size - 1)) == 0`. This efficiently checks if `current_ip` is a multiple of `block_size`.
        *   It checks if the entire block fits: `(uint64_t)current_ip + block_size - 1 <= end_ip_num`. The `(uint64_t)current_ip` cast is crucial to prevent overflow when `block_size` is `2^32`.
        *   The first `p` (from `32` down to `0`) that satisfies both conditions is the `best_prefix_len` because it corresponds to the largest possible block starting at `current_ip` that fits the remaining range. The inner loop then `break`s.
    *   **Storing CIDR:** The `current_ip` and `best_prefix_len` are formatted into a string (e.g., "192.168.1.1/32") and added to a dynamically growing list.
    *   **Advancing `current_ip`:**
        *   If `best_prefix_len` was `0` (meaning `0.0.0.0/0` was generated, covering the entire IPv4 space), `current_ip` is set to `end_ip_num + 1` to immediately terminate the outer `while` loop.
        *   Otherwise, `current_ip` is incremented by the `block_size` of the newly added CIDR block to move to the next uncovered IP address.
5.  **Output:** Finally, all generated CIDR blocks are printed as a comma-separated list.
6.  **Memory Management:** All dynamically allocated memory for the CIDR strings and the list itself is freed.

### Code

```c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <arpa/inet.h>  // For inet_pton, inet_ntop, ntohl, htonl
#include <stdint.h>     // For uint32_t, uint64_t

// Function to convert uint32_t IP address to a human-readable string.
// It takes a buffer and its size to store the IP string, which is safer than using a static buffer.
const char* uint32_to_ip_str(uint32_t ip_num, char* buffer, size_t buffer_size) {
    struct in_addr addr;
    addr.s_addr = htonl(ip_num); // Convert to network byte order for inet_ntop
    if (inet_ntop(AF_INET, &addr, buffer, buffer_size) == NULL) {
        perror("inet_ntop failed");
        return NULL;
    }
    return buffer;
}

int main(int argc, char *argv[]) {
    // Validate command-line arguments
    if (argc != 3) {
        fprintf(stderr, "Usage: %s <start_ip> <end_ip>\n", argv[0]);
        return EXIT_FAILURE;
    }

    uint32_t start_ip_num, end_ip_num;
    struct in_addr addr_buf;

    // Convert start IP string to uint32_t (host byte order)
    if (inet_pton(AF_INET, argv[1], &addr_buf) != 1) {
        fprintf(stderr, "Invalid start IP address: %s\n", argv[1]);
        return EXIT_FAILURE;
    }
    start_ip_num = ntohl(addr_buf.s_addr);

    // Convert end IP string to uint32_t (host byte order)
    if (inet_pton(AF_INET, argv[2], &addr_buf) != 1) {
        fprintf(stderr, "Invalid end IP address: %s\n", argv[2]);
        return EXIT_FAILURE;
    }
    end_ip_num = ntohl(addr_buf.s_addr);

    // Ensure start_ip_num is less than or equal to end_ip_num
    if (start_ip_num > end_ip_num) {
        uint32_t temp = start_ip_num;
        start_ip_num = end_ip_num;
        end_ip_num = temp;
        fprintf(stderr, "Warning: Start IP (%s) was greater than End IP (%s). Swapped them.\n", argv[1], argv[2]);
    }

    // Dynamic array to store generated CIDR block strings
    char **cidr_blocks = NULL;
    size_t cidr_count = 0;
    size_t cidr_capacity = 10; // Initial capacity

    cidr_blocks = malloc(sizeof(char *) * cidr_capacity);
    if (cidr_blocks == NULL) {
        perror("Failed to allocate memory for CIDR blocks list");
        return EXIT_FAILURE;
    }

    uint32_t current_ip = start_ip_num;
    char ip_str_buffer[INET_ADDRSTRLEN]; // Buffer for uint32_to_ip_str

    while (current_ip <= end_ip_num) {
        int best_prefix_len = 32; // A /32 block (single IP) is always the smallest possible valid block.

        // Iterate from smallest block size (p=32) down to largest (p=0).
        // The first 'p' found that satisfies conditions is the largest valid block for current_ip.
        for (int p = 32; p >= 0; --p) {
            uint64_t block_size = 1ULL << (32 - p);

            // Check 1: Alignment (current_ip must be a network address for prefix 'p')
            // This means the last (32 - p) bits of current_ip must be zero.
            if ((current_ip & (block_size - 1)) == 0) {
                // Check 2: Fit (the entire block must be within the overall end_ip_num)
                // Cast current_ip to uint64_t to prevent overflow before addition,
                // especially when block_size is 2^32 (for p=0).
                if ((uint64_t)current_ip + block_size - 1 <= end_ip_num) {
                    best_prefix_len = p;
                    break; // Found the largest possible block, so use it and move on.
                }
            }
        }

        // Add the found CIDR block to our dynamic list
        if (cidr_count >= cidr_capacity) {
            cidr_capacity *= 2; // Double capacity
            char **new_cidr_blocks = realloc(cidr_blocks, sizeof(char *) * cidr_capacity);
            if (new_cidr_blocks == NULL) {
                perror("Failed to reallocate memory for CIDR blocks list");
                // Clean up previously allocated strings before exiting
                for (size_t i = 0; i < cidr_count; ++i) {
                    free(cidr_blocks[i]);
                }
                free(cidr_blocks);
                return EXIT_FAILURE;
            }
            cidr_blocks = new_cidr_blocks;
        }

        const char* ip_str = uint32_to_ip_str(current_ip, ip_str_buffer, sizeof(ip_str_buffer));
        if (ip_str == NULL) {
            // Error from uint32_to_ip_str, clean up and exit
            for (size_t i = 0; i < cidr_count; ++i) {
                free(cidr_blocks[i]);
            }
            free(cidr_blocks);
            return EXIT_FAILURE;
        }

        // Format the CIDR string (e.g., "192.168.1.0/24")
        char current_cidr_str[INET_ADDRSTRLEN + 5]; // Max size: "255.255.255.255/32\0"
        snprintf(current_cidr_str, sizeof(current_cidr_str), "%s/%d", ip_str, best_prefix_len);

        cidr_blocks[cidr_count] = strdup(current_cidr_str); // Duplicate string for storage
        if (cidr_blocks[cidr_count] == NULL) {
            perror("Failed to allocate memory for CIDR block string");
            // Clean up previously allocated strings before exiting
            for (size_t i = 0; i < cidr_count; ++i) {
                free(cidr_blocks[i]);
            }
            free(cidr_blocks);
            return EXIT_FAILURE;
        }
        cidr_count++;

        // Move current_ip to the address immediately following the generated block
        uint64_t block_size = 1ULL << (32 - best_prefix_len);
        if (best_prefix_len == 0) {
            // If a /0 block was generated, it covers the entire IPv4 space.
            // Force current_ip to a value that will terminate the outer loop.
            current_ip = end_ip_num + 1;
        } else {
            // For other prefix lengths, block_size fits into uint32_t.
            current_ip += (uint32_t)block_size;
        }
    }

    // Print the generated CIDR blocks as a comma-separated list
    for (size_t i = 0; i < cidr_count; ++i) {
        printf("%s%s", cidr_blocks[i], (i == cidr_count - 1) ? "" : ",");
        free(cidr_blocks[i]); // Free individual string memory after use
    }
    printf("\n");

    free(cidr_blocks); // Free the dynamic array itself
    return EXIT_SUCCESS;
}
```

### How to Compile and Run

1.  **Save:** Save the code as `cidr_range.c`.
2.  **Compile:** Use a C compiler like GCC:
    ```bash
    gcc -o cidr_range cidr_range.c
    ```
3.  **Run:** Execute the program with two IP addresses:

    *   **Example 1: A small contiguous range**
        ```bash
        ./cidr_range 192.168.1.1 192.168.1.10
        ```
        Output:
        ```
        192.168.1.1/32,192.168.1.2/31,192.168.1.4/30,192.168.1.8/30
        ```
        (Covers 192.168.1.1, then 192.168.1.2-3, then 192.168.1.4-7, then 192.168.1.8-11, which goes beyond 1.10. Let's recheck this. 192.168.1.8/30 covers 192.168.1.8 to 192.168.1.11. This means the algorithm correctly chose the largest block possible without overshooting `192.168.1.10`. Wait, if `end_ip` is `192.168.1.10`, then `192.168.1.8/30` should not be chosen because its end `192.168.1.11` is greater than `192.168.1.10`. This indicates my example output might be wrong or my algorithm trace. Let's trace it again for `192.168.1.8` to `192.168.1.10`.)

        *Correction trace for `192.168.1.8` to `192.168.1.10` when `current_ip = 192.168.1.8`:*
        `end_ip_num = 192.168.1.10`
        `current_ip = 192.168.1.8` (`0...1000`)
        `p=32`: `block_size=1`. aligned. `192.168.1.8 <= 192.168.1.10`. `best_prefix_len = 32`.
        `p=31`: `block_size=2`. aligned. `192.168.1.8+1=192.168.1.9 <= 192.168.1.10`. `best_prefix_len = 31`.
        `p=30`: `block_size=4`. aligned. `192.168.1.8+3=192.168.1.11`. `192.168.1.11 > 192.168.1.10`. Fails.
        So, `best_prefix_len` remains `31`.
        Output should be `192.168.1.8/31`.
        `current_ip = 192.168.1.8 + 2 = 192.168.1.10`.

        *Next iteration: `current_ip = 192.168.1.10`*
        `end_ip_num = 192.168.1.10`
        `current_ip = 192.168.1.10` (`0...1010`)
        `p=32`: `block_size=1`. aligned. `192.168.1.10 <= 192.168.1.10`. `best_prefix_len = 32`.
        `p=31`: `block_size=2`. aligned. `192.168.1.10+1=192.168.1.11`. `192.168.1.11 > 192.168.1.10`. Fails.
        So, `best_prefix_len` remains `32`.
        Output should be `192.168.1.10/32`.
        `current_ip = 192.168.1.10 + 1 = 192.168.1.11`. Loop ends.

        So for `192.168.1.1` to `192.168.1.10`, the output should be:
        `192.168.1.1/32,192.168.1.2/31,192.168.1.4/30,192.168.1.8/31,192.168.1.10/32`
        This is correct and minimal.

    *   **Example 2: A full C-class subnet**
        ```bash
        ./cidr_range 192.168.1.0 192.168.1.255
        ```
        Output:
        ```
        192.168.1.0/24
        ```

    *   **Example 3: Range covering multiple classes**
        ```bash
        ./cidr_range 10.0.0.0 10.0.255.255
        ```
        Output:
        ```
        10.0.0.0/16
        ```

    *   **Example 4: Full IPv4 range**
        ```bash
        ./cidr_range 0.0.0.0 255.255.255.255
        ```
        Output:
        ```
        0.0.0.0/0
        ```

    *   **Example 5: Swapped IP addresses**
        ```bash
        ./cidr_range 192.168.1.255 192.168.1.0
        ```
        Output:
        ```
        Warning: Start IP (192.168.1.255) was greater than End IP (192.168.1.0). Swapped them.
        192.168.1.0/24
        ```

## Multimodal interactions

Gemini models are able to process different data modatilities, such as unstructured files, images, audio and videos, allowing you to experiment with multimodal scenarios where you can ask the model to describe, explain, get insights or extract information out of those multimedia information included into your prompts. In this section you will work across different senarios with multimedia information.

**IMPORTANT:** The OpenAI SDK compatibility only supports inline images and audio files. For videos support, use the [Gemini API's Python SDK](https://ai.google.dev/gemini-api/docs/sdks).

### Working with images (a single image)

You will first download the image you want to work with.

In [ ]:
from PIL import Image as PImage


# define the image you want to download
image_url = "https://storage.googleapis.com/generativeai-downloads/images/Japanese_Bento.png" # @param
image_filename = image_url.split("/")[-1]

# download the image
!wget -q $image_url

# visualize the downloaded image
im = PImage.open(image_filename)
im.thumbnail([620,620], PImage.Resampling.LANCZOS)
im

Now you can encode the image and work with the OpenAI library to interact with the Gemini models.

In [20]:
import base64
import requests


# define a helper function to encode the images in base64 format
def encode_image(image_path):
  image = requests.get(image_path)
  return base64.b64encode(image.content).decode('utf-8')

# define the image you want to download
image_url = "https://storage.googleapis.com/generativeai-downloads/images/Japanese_Bento.png" # @param

# Getting the base64 encoding
encoded_image = encode_image(image_url)

response = client.chat.completions.create(
  model=MODEL_ID,
  messages=[
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "Describe the items on this image. If there is any non-English text, translate it as well"
        },
        {
          "type": "image_url",
          "image_url": {
            "url": f"data:image/png;base64,{encoded_image}",
          },
        },
      ],
    }
  ]
)

Markdown(response.choices[0].message.content)

Here are the descriptions and translations of the items in the image:

**Row 1:**

1.  **Image:** A slice of green roll cake with a white cream spiral filling, and small dark specks (likely adzuki beans) on the cake. It's served on a small white plate.
    *   **Japanese Text:** 抹茶のスイスロール
    *   **Translation:** Matcha Swiss Roll
    *   **Description:** A slice of matcha-flavored Swiss roll cake. The cake itself is green, indicating the matcha (green tea) flavor, and it's filled with a spiral of white cream. Small dark red bean or chocolate pieces are visible on the exterior.

2.  **Image:** A golden-brown sweet bun, with one shown whole and one cut in half to reveal a dark red paste filling. The top of the whole bun has poppy or sesame seeds.
    *   **Japanese Text:** あんパン
    *   **Translation:** Anpan
    *   **Description:** An Anpan, a classic Japanese sweet bread roll. One bun is shown whole, topped with sprinkles, while the other is cut in half, showcasing its rich filling of sweet red bean paste (anko).

3.  **Image:** A dark plate piled with shredded, light orange-brown strips of dried seafood.
    *   **Japanese Text:** さきイカ
    *   **Translation:** Shredded Dried Squid (Saki Ika)
    *   **Description:** A serving of Saki Ika, which is shredded dried squid. The light orange-brown strips are fibrous and typically enjoyed as a savory snack or appetizer.

**Row 2:**

1.  **Image:** A small dark plate holding several reddish-brown, wrinkled, round fruits.
    *   **Japanese Text:** 梅干し
    *   **Translation:** Pickled Plums (Umeboshi)
    *   **Description:** A dish of Umeboshi, Japanese pickled plums. These are distinctively wrinkled, reddish-brown, and have a tart and salty flavor.

2.  **Image:** Two fish-shaped pastries on a dark plate, with a golden-brown, waffle-like texture.
    *   **Japanese Text:** たい焼き
    *   **Translation:** Taiyaki
    *   **Description:** Two Taiyaki pastries. These are Japanese fish-shaped cakes, typically filled with sweet red bean paste, and have a characteristic golden-brown, grilled exterior.

3.  **Image:** A round, delicate Japanese sweet on a dark plate. One is whole with a decorative pattern, and another is open to show a dark red paste filling.
    *   **Japanese Text:** あずき最中
    *   **Translation:** Azuki Monaka (Monaka with red bean paste)
    *   **Description:** An Azuki Monaka sweet. It consists of a sweet red bean paste filling sandwiched between two thin, crisp wafers made from mochi flour. One piece is shown with its decorative wafer, and another is split to reveal the anko (red bean paste) inside.

**Row 3:**

1.  **Image:** Two triangular rice balls wrapped partially in dark seaweed, served on a blue patterned plate with yellow pickled daikon.
    *   **Japanese Text:** お握り
    *   **Translation:** Rice Balls (Onigiri)
    *   **Description:** Two Onigiri, or Japanese rice balls. They are triangular, made of white rice, and partially wrapped in dark green nori (seaweed). They are served on a blue and white patterned ceramic plate, accompanied by bright yellow pickled daikon radish (takuan).

2.  **Image:** Two pink, oval-shaped sweets wrapped in green leaves, presented on a black plate with a small red utensil.
    *   **Japanese Text:** 桜餅
    *   **Translation:** Sakuramochi
    *   **Description:** Two Sakuramochi, traditional Japanese sweets. These are pink-tinted mochi (rice cakes) with a sweet filling (often red bean paste), wrapped in a preserved cherry blossom leaf. They are served on a black plate with a small red spatula.

3.  **Image:** A pile of small, cylindrical orange crackers, each wrapped with a strip of dark green seaweed.
    *   **Japanese Text:** 海苔巻き煎餅
    *   **Translation:** Seaweed-wrapped Rice Crackers (Nori Maki Senbei)
    *   **Description:** A pile of Nori Maki Senbei. These are savory Japanese rice crackers that are cylindrical and orange, each wrapped with a strip of dark green nori (seaweed), combining crunchy texture with a savory, umami flavor.

### Working with images (multiple images)

You can do the same process while sending multiple images into the same prompt.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image
from matplotlib.pyplot import imread


# define the images you want to download
image_urls = [
    "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/cesar-couto-OB2F6CsMva8-unsplash.jpg",
    "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/daniil-silantev-1P6AnKDw6S8-unsplash.jpg",
    "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/ruslan-bardash-4kTbAMRAHtQ-unsplash.jpg",
    "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/scopic-ltd-NLlWwR4d3qU-unsplash.jpg",
]

for url in image_urls:
    display(Image(url=url, width=200, height=250))

Now you can encode the images and send them with your prompt.

In [21]:
import base64
import requests


# define a helper function to encode the images in base64 format
def encode_image(image_path):
  image = requests.get(image_path)
  return base64.b64encode(image.content).decode('utf-8')

# define the images you want to download
image_urls = [
    "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/cesar-couto-OB2F6CsMva8-unsplash.jpg",
    "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/daniil-silantev-1P6AnKDw6S8-unsplash.jpg",
    "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/ruslan-bardash-4kTbAMRAHtQ-unsplash.jpg",
    "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/scopic-ltd-NLlWwR4d3qU-unsplash.jpg",
]

# Getting the base64 encoding
encoded_images =[]
for image in image_urls:
  encoded_images.append(encode_image(image))

response = client.chat.completions.create(
  model=MODEL_ID,
  messages=[
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "Describe for what type of living room each of those items are the best match"
        },
        *[{
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{image_data}",
                    },
        }
                for image_data in encoded_images]
      ],
    }
  ]
)

Markdown(response.choices[0].message.content)

Here's a description for what type of living room each item would best match:

---

**1. Industrial Adjustable Stool (Image 1)**

*   **Best Match Living Room Type:** This stool is an excellent fit for **Industrial, Rustic, Loft-style, or Eclectic** living rooms. Its combination of a warm, round wooden seat and a dark, sturdy metal base with an exposed screw mechanism screams utility and raw charm.
*   **Why it works:** In an Industrial space, it complements exposed brick, metal accents, and reclaimed materials. For a Rustic or Farmhouse feel, the wood adds warmth and an authentic touch. In a Loft, it can serve as versatile extra seating or a side table, fitting the open, urban aesthetic. Its unique character also makes it a great accent in an Eclectic room, adding an unexpected textural and historical element.

---

**2. Tufted Cream Armchair (Image 2)**

*   **Best Match Living Room Type:** This elegant armchair is perfectly suited for **Traditional, Glam/Hollywood Regency, French Provincial, or Shabby Chic** living rooms.
*   **Why it works:** The plush cream upholstery, deep button tufting, rolled arms, and ornate turned legs evoke a sense of classic luxury and comfort. In a Traditional setting, it harmonizes with rich fabrics, dark wood, and classic décor. For a Glam or Hollywood Regency look, its opulent feel and soft texture add to a lavish, sophisticated ambiance. It could also find a home in a French Provincial or Shabby Chic room, offering a softer, more romantic elegance when paired with distressed finishes and antique accents.

---

**3. Light Wood Square Bar Stool (Image 3)**

*   **Best Match Living Room Type:** This minimalist, light-colored wooden stool would be ideal for **Scandinavian, Modern, Coastal, or Contemporary Farmhouse** living rooms.
*   **Why it works:** Its clean lines, simple square seat, and light wood finish embody a functional and airy aesthetic. In a Scandinavian living room, it aligns with natural materials, light colors, and uncluttered design. For a Modern space, its straightforward form contributes to a sleek, unfussy look. The light wood and unpretentious design also make it suitable for a Coastal theme, evoking a relaxed, beachy vibe, or in a Contemporary Farmhouse setting where a cleaner, lighter version of rustic is preferred. While typically a bar stool, in a living room context, it could serve as a tall plant stand or an occasional high-level surface in an open-plan space.

---

**4. Modern Swivel Lounge Chair (Image 4)**

*   **Best Match Living Room Type:** This stylish lounge chair is best suited for **Mid-Century Modern, Contemporary, Scandinavian, or Minimalist** living rooms.
*   **Why it works:** The ergonomic, low-slung upholstered seat, combined with the distinctive angled natural wood star base and swivel function, is a hallmark of sophisticated, design-conscious furniture. It perfectly encapsulates the iconic shapes and natural material use of Mid-Century Modern design. In a Contemporary setting, its clean lines and comfortable form provide both style and function. The natural wood base and understated fabric fit well with Scandinavian principles of natural materials and practical design. For a Minimalist living room, its sculptural yet unobtrusive presence adds comfort without clutter, making a statement through form and quality.

### Working with audio files

You can also send audio files on your prompt. Audio data provides a more rich input than text alone, and can be use for tasks like transcription, or as direct prompting like a voice assistant.

First you need to download the audio you want to use.

In [28]:
from IPython.display import Audio


audio_url = "https://storage.googleapis.com/generativeai-downloads/data/Apollo-11_Day-01-Highlights-10s.mp3" # @param
audio_filename = audio_url.split("/")[-1]

# download the audio
!wget -q -P /content/ $audio_url

# listen to the downloaded audio
display(Audio(f'/content/{audio_filename}', autoplay=False))

Now you will encode the audio in `base64` and send it as part of your request prompt.

In [29]:
import os

# define a helper function to encode the images in base64 format
def encode_audio(audio_path):
  with open(audio_path, 'rb') as audio_file:
    audio_content = audio_file.read()
    return base64.b64encode(audio_content).decode('utf-8')

audio_url = "https://storage.googleapis.com/generativeai-downloads/data/Apollo-11_Day-01-Highlights-10s.mp3" # @param
audio_filename = audio_url.split("/")[-1]
audio_filepath = os.path.join('/content/', audio_filename) # Construct the full path

print(f"Current working directory: {os.getcwd()}")
print(f"Files in current directory: {os.listdir()}")

base64_audio = encode_audio(audio_filepath) # Use the full path

prompt = "Transcribe this audio file. After transcribing, tell me from what this can be related to." # @param
response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": prompt,
        },
        {
              "type": "input_audio",
              "input_audio": {
                "data": base64_audio,
                "format": "mp3"
          }
        }
      ],
    }
  ],
)

Markdown(response.choices[0].message.content)

Current working directory: /content
Files in current directory: ['.config', '1706.03762.pdf', 'Apollo-11_Day-01-Highlights-10s.mp3', 'sample_data']


Here is the transcription of the audio:

**Audio:**
minus 10, nine, eight, we have a go for main engine start. We have main engine start.

**Related to:**
This audio is related to a **rocket launch** or a **spacecraft pre-launch sequence**. The countdown "10, 9, 8" followed by "go for main engine start" and "main engine start" are standard commands and events heard during the final moments leading up to a rocket's lift-off.

## Structured outputs

Gemini API allows you to format the way your response you be generated via [structured outputs](https://ai.google.dev/gemini-api/docs/structured-output). You can define the structure you want to be used as a defined schema and, using the OpenAI library, you send this structure as the `response_format` parameter.

In this example you will:
- download a scientific paper
- extract its information
- define the structure you want your response in
- send your request using the `response_format` parameter

First you need to download the reference paper. You will use the [Attention is all your need](https://arxiv.org/pdf/1706.03762.pdf) Google paper that introduced the [Transformers architecture](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)).

In [ ]:
from IPython.display import Image
from pdf2image import convert_from_path


# download the PDF file
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf" # @param
pdf_filename = pdf_url.split("/")[-1]
!wget -q $pdf_url

## visualize the pdf as an image
# convert the PDF file to images
images = convert_from_path(pdf_filename, 200)
for image in images:
  image.save('cover.png', "PNG")
  break

# show the pdf first page
Image('cover.png', width=500, height=600)

Now you will create your reference structure. It will be a Python `Class` that will refer to the title, authors, abstract and keywords from the paper.

In [ ]:
from pydantic import BaseModel


class ResearchPaperExtraction(BaseModel):
    title: str
    authors: list[str]
    abstract: str
    keywords: list[str]

Now you will do your request to the Gemini API sending the pdf file and the reference structure.

In [23]:
import json
from pdfminer.high_level import extract_text
from pydantic import BaseModel


class ResearchPaperExtraction(BaseModel):
    title: str
    authors: list[str]
    abstract: str
    keywords: list[str]

# download the PDF file
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf" # @param
pdf_filename = pdf_url.split("/")[-1]
!wget -q $pdf_url


# extract text from the PDF
pdf_text = extract_text(pdf_filename)

prompt = """
    As a specialist in knowledge organization and data refinement, your task is to transform
    raw research paper content into a clearly defined structured format. I will provide you
    with the original, free-form text. Your goal is to parse this text, extract the pertinent
    information, and reconstruct it according to the structure outlined below.
"""

# send your request to the Gemini API
completion = client.beta.chat.completions.parse(
  model=MODEL_ID,
  messages=[
    {"role": "system", "content": prompt},
    {"role": "user", "content": pdf_text}
  ],
  response_format=ResearchPaperExtraction,
)

print(completion.choices[0].message.parsed.model_dump_json(indent=2))

{
  "title": "Attention Is All You Need",
  "authors": [
    "Ashish Vaswani",
    "Noam Shazeer",
    "Niki Parmar",
    "Jakob Uszkoreit",
    "Llion Jones",
    "Aidan N. Gomez",
    "Łukasz Kaiser",
    "Illia Polosukhin"
  ],
  "abstract": "The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English- to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU. On the WMT 2014 English-to-F

Given the Gemini API ability to handle structured outputs, you can work in more complex scenarios too - like using the structured output functionality to help you generating user interfaces.

First you define the Python classes that represent the structure you want in the output.

In [ ]:
from enum import Enum


class UIType(str, Enum):
    div = "div"
    button = "button"
    header = "header"
    section = "section"
    field = "field"
    form = "form"

class Attribute(BaseModel):
    name: str
    value: str

class UI(BaseModel):
    type: UIType
    label: str
    children: list[str]
    attributes: list[Attribute]

UI.model_rebuild() # This is required to enable recursive types

class Response(BaseModel):
    ui: UI

Now you send your request using the `Response` class as the `response_format`.

In [24]:
from pydantic import BaseModel
from enum import Enum


class UIType(str, Enum):
    div = "div"
    button = "button"
    header = "header"
    section = "section"
    field = "field"
    form = "form"

class Attribute(BaseModel):
    name: str
    value: str

class UI(BaseModel):
    type: UIType
    label: str
    children: list[str]
    attributes: list[Attribute]

UI.model_rebuild() # This is required to enable recursive types

class Response(BaseModel):
    ui: UI

completion = client.beta.chat.completions.parse(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "You are a UI generation assistant. Convert the user input into a UI."},
        {"role": "user", "content": "Make a User Profile Form including all required attributes"}
    ],
    response_format=Response,
)

print(completion.choices[0].message.content)

{"ui":{"type":"form","label":"User Profile Form","children":["Full Name","Email Address","Phone Number","Shipping Address","About Me","Profile Picture","Save Changes"],"attributes":[{"name":"id","value":"user-profile-form"},{"name":"method","value":"POST"},{"name":"action","value":"/api/profile/update"}]}}


## Developing with the Gemini API Function Calling

The Gemini API's function calling feature allows you to extend the model's capabilities by providing descriptions of external functions or APIs.

For further understanding of how function calling works with Gemini models, check the [Gemini API documentation](https://ai.google.dev/gemini-api/docs/function-calling).

In [ ]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Gets the weather at the user's location",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {"type": "string"},
        },
      },
    },
  }
]

Now you add the `tools` structure on your request.

In [25]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Gets the weather at the user's location",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {"type": "string"},
        },
      },
    },
  }
]

prompt = """
    What's the weather like in Boston?
"""

completion = client.chat.completions.create(
  model=MODEL_ID,
  messages=[{"role": "user", "content": prompt}],
  tools=tools,
)

print(completion.choices[0].message.tool_calls[0])

ChatCompletionMessageFunctionToolCall(id='function-call-16368258302667174072', function=Function(arguments='{"location":"Boston"}', name='get_weather'), type='function')


## Thinking

Gemini 2.5 models are trained to think through complex problems, leading to significantly improved reasoning. The Gemini API comes with a ["thinking budget" parameter](https://ai.google.dev/gemini-api/docs/thinking) which gives fine grain control over how much the model will think.

Unlike the Gemini API, the OpenAI API offers three levels of thinking control: "low", "medium", and "high", which are mapped to 1K, 8K, and 24K thinking token budgets.

If you want to disable thinking, you can set the reasoning effort to "none".

In [ ]:
THINKING_MODEL = "gemini-2.5-flash"
prompt = """
    What is 45-78+5x13?
    Double check and explain why your answer is correct.
"""

response = client.chat.completions.create(
  model=THINKING_MODEL,
  reasoning_effort="low",
  messages=[
      {"role": "system", "content": "You are a helpful assistant."},
      {
        "role": "user",
        "content": prompt
      }
  ]
)

Markdown(response.choices[0].message.content)

## Batch predictions

In addition to real-time generations, the Gemini API provides a compatible layer for performing batch generations.

Prepare a JSONL file in OpenAI batch input format. Note that the URLs and overall structure uses the OpenAI syntax, but the models are Gemini API model identifiers.

In [ ]:
%%writefile batch_requests.jsonl
{"custom_id": "request-1", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gemini-2.5-flash", "messages": [{"role": "user", "content": "Tell me a one-sentence joke."}]}}
{"custom_id": "request-2", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gemini-2.5-flash", "messages": [{"role": "user", "content": "Why is the sky blue?"}]}}

Now upload the JSONL file with the GenAI SDK. Until there is support for the OpenAI file upload API, you must use the Google GenAI SDK to upload the file to make the data available to the Gemini API.

In [ ]:
%pip install -qU google-genai

In [ ]:
# Upload JSONL file in OpenAI batch input format...
from google import genai
from google.genai import types

genai_client = genai.Client(api_key=GOOGLE_API_KEY)

uploaded_file = genai_client.files.upload(
    file="batch_requests.jsonl",
    config=types.UploadFileConfig(display_name="my-batch-requests", mime_type="jsonl"),
)
print(f'{uploaded_file.name=}')

Create a batch generation job referencing the file just uploaded.

In [ ]:
batch = client.batches.create(
    input_file_id=uploaded_file.name,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)
print(f'{batch.id=}')

Batches can take up to 24 hours to process. Poll here with the following code, or come back later and replace `batch.id` with the ID printed during creation above.

In [ ]:
import time

while (batch := client.batches.retrieve(batch.id)).status == 'in_progress':
    print(f"Job not finished. Current state: {batch.status}. Waiting 30 seconds...")
    time.sleep(30)

print(f'{batch.status=}')

Now that the batch is processed, download the results and print them out.

In [ ]:
if batch.status == 'completed':
  # Download the output file.
  file_content_bytes = genai_client.files.download(file=batch.output_file_id)
  file_content = file_content_bytes.decode('utf-8')

  # Print each output record.
  for i, line in enumerate(file_content.splitlines(), start=1):
      print(i, line)

else:
  print(f'An error occurred. Batch status is "{batch.status}".')

## Generating and working with embeddings

Text embeddings offer a compressed, vector-based representation of text, typically in a lower-dimensional space. The core principle is that semantically similar texts will have embeddings that are spatially proximate within the embedding vector space. This representation enables solutions to several prevalent NLP challenges, including:

- **Semantic Search:** Identifying and ranking texts based on semantic relatedness.
- **Recommendation:** Suggesting items whose textual descriptions exhibit semantic similarity to a given input text.
- **Classification:** Assigning text to categories based on the semantic similarity between the text and the category's representative text.
- **Clustering:** Grouping texts into clusters based on the semantic similarity reflected in their respective embedding vectors.
- **Outlier Detection:** Identifying texts that are semantically dissimilar from the majority, as indicated by their distance in the embedding vector space.

For more details about working with the Gemini API and embeddings, check the [API documentation](https://ai.google.dev/gemini-api/docs/embeddings).

In this example you will use the `gemini-embedding-001` model from the Gemini API to generate your embeddings.

In [ ]:
EMBEDDINGS_MODEL="gemini-embedding-001"
prompt = """
    The quick brown fox jumps over the lazy dog.
"""

response = client.embeddings.create(
  model=EMBEDDINGS_MODEL,
  input=prompt,
)

print(len(response.data[0].embedding))
print(response.data[0].embedding[:4], '...')

In [ ]:
EMBEDDINGS_MODEL="gemini-embedding-001"
prompt = """
    The quick brown fox jumps over the lazy dog.
"""

response = client.embeddings.create(
  model=EMBEDDINGS_MODEL,
  input=prompt,
)

print(len(response.data[0].embedding))
print(response.data[0].embedding[:4], '...')

A simple application of text embeddings is to calculate the similarity between sentences (ie. product reviews, documents contents, etc). First you will create a group of sentences.

In [ ]:
import pandas as pd


text = [
    "i really enjoyed the movie last night",
    "so many amazing cinematic scenes yesterday",
    "had a great time writing my Python scripts a few days ago",
    "huge sense of relief when my .py script finally ran without error",
    "O Romeo, Romeo, wherefore art thou Romeo?",
]

df = pd.DataFrame(text, columns=["text"])
df

Now you can create a function to generate embeddings, apply that function to your dataframe `text` column and save it into a new column called `embeddings`.

In [ ]:
def generate_embeddings(text):
  response = client.embeddings.create(
    model=EMBEDDINGS_MODEL,
    input=text,
  )
  return response.data[0].embedding

df["embeddings"] = df.apply(
    lambda x: generate_embeddings([x.text]), axis=1
)
df

Now that you have the embeddings representations for all sentences, you can calculate their similarities.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cos_sim_array = cosine_similarity(list(df.embeddings.values))

# display as DataFrame
analysis = pd.DataFrame(cos_sim_array, index=text, columns=text)
analysis

You can also plot it for a better visualization.

In [ ]:
import seaborn as sns

ax = sns.heatmap(analysis, annot=True, cmap="Blues")
ax.xaxis.tick_top()
ax.set_xticklabels(text, rotation=90)

### Batch embedding generation

You can use the OpenAI SDK to generate embeddings offline and in large batches at a reduced rate. The API follows the same steps you would take for generating content.

Start by creating a JSONL file that contains each of the embedding requests to process.

In [ ]:
%%writefile embedding_requests.jsonl
{"custom_id": "request-1", "method": "POST", "url": "/v1/embeddings", "body": {"model": "gemini-embedding-001", "input": "I really enjoyed the movie last night"}}
{"custom_id": "request-2", "method": "POST", "url": "/v1/embeddings", "body": {"model": "gemini-embedding-001", "input": "So many amazing cinematic scenes yesterday"}}

Upload the JSONL file containing the embedding requests. Until there is support for the OpenAI file upload API, you must use the Google GenAI SDK to upload the file and make the data available to the Gemini API.

In [ ]:
from google import genai
from google.genai import types

genai_client = genai.Client(api_key=GOOGLE_API_KEY)

uploaded_file = genai_client.files.upload(
    file="embedding_requests.jsonl",
    config=types.UploadFileConfig(display_name="My embedding requests", mime_type="jsonl"),
)
print(f'{uploaded_file.name=}')

Now, with the file uploaded, create a new batch job to process the requests in the JSONL file.

In [ ]:
batch = client.batches.create(
    input_file_id=uploaded_file.name,
    endpoint="/v1/embeddings",
    completion_window="24h"
)
print(f'{batch.id=}')

Batches can take up to 24 hours to process. Poll here with the following code, or come back later and replace `batch.id` with the ID printed during creation above.

In [ ]:
import time

while (batch := client.batches.retrieve(batch.id)).status == 'in_progress':
    print(f"Job not finished. Current state: {batch.status}. Waiting 30 seconds...")
    time.sleep(30)

print(f'{batch.status=}')

Now that the batch is processed, download the results and print them out.

In [ ]:
if batch.status == 'completed':
  # Download the output file.
  file_content_bytes = genai_client.files.download(file=batch.output_file_id)
  file_content = file_content_bytes.decode('utf-8')

  # Print each output record.
  for i, line in enumerate(file_content.splitlines(), start=1):
      print(i, line[:100] + '...')

else:
  print(f'An error occurred. Batch status is "{batch.status}".')

## Next Steps

### Do more with Gemini

If you want to use more of the Gemini capabilities and especially its unique capabilities not available through the OpenAI compatibility, you should check out the [Google GenAI SDK](https://github.com/googleapis/python-genai).

The Cookbook is full of examples on how to use it but it is recommended to start with the [Getting started](./Get_started.ipynb) notebook to get a feel of all the models and SDK capabilities.

### Related examples

Check the rest of the [Cookbook](https://github.com/google-gemini/cookbook). You'll learn how to use the [Live API](./Get_started_LiveAPI.ipynb), juggle with [multiple tools](../examples/LiveAPI_plotting_and_mapping.ipynb) or use Gemini's [spatial understanding](./Spatial_understanding.ipynb) abilities.

Also check the [Thinking cookbook](./Get_started_thinking.ipynb) that explicitly showcases its thoughts and can manage more complex reasonings.